# Dataiku · 중계서버 연결 → 법제처 호출

이 노트북 하나만 가져와 위에서 아래로 실행합니다.
**1단계:** 중계 주소에 HEAD 요청 → HTTP 응답 확인. **2단계:** 같은 중계를 통해 법령 XML 본문 GET 요청.
법제처·중계 인증은 2단계에서만 입력합니다. 기존 노트북이나 별도 모듈 업로드는 필요 없습니다.


## 준비 · 주소와 조회할 법령
Python 3.9 이상과 requests가 있는 Dataiku 코드 환경을 사용하세요.
같은 run_id는 저장된 결과를 보여줍니다. 연결 설정을 고친 뒤 다시 확인할 때 run_id를 바꾸세요.


In [ ]:
# 기존 사진의 개발계 주소입니다. 실제 사용할 주소가 다르면 바꾸세요.
CONFIG = {
    "relay_origin": "https://45.101.64.86:8443",
    "law_origin": "http://www.law.go.kr",
    "law_id": "010513",
    "law_name": "자본시장과 금융투자업에 관한 법률",
    "target_in_header": True,
    "ca_bundle": "",  # 사내 CA가 필요하면 실행 서버의 PEM 파일 경로
    "output_root": "dataiku_relay_law_results",
    "run_id": "relay-law-01",  # 새로 요청하려면 02, 03처럼 변경. 기존 기록은 보존됩니다.
}


### 공통 코드 · 수정 없이 한 번 실행
요청·검증·저장 함수입니다. 이 셀 자체는 네트워크 요청을 보내지 않습니다.


In [ ]:
"""Dataiku 법령 API 수신 진단. 수집/MAIN 등록 도구가 아닌 독립 현장 확인 코드."""
import csv
import gzip
import hashlib
import io
import json
import os
import platform
import re
import sys
import time
import uuid
from pathlib import Path
from urllib.parse import quote, quote_plus, urlsplit
import xml.etree.ElementTree as ET

import requests

VERSION = "dataiku-law-api-20261015-r1"
MAX_BYTES = 12 * 1024 * 1024
MAX_CALLS = 8


def digest(raw):
    return hashlib.sha256(raw).hexdigest()


def json_bytes(obj):
    return (json.dumps(obj, ensure_ascii=False, indent=2) + "\n").encode("utf-8")


def masked(text, secrets=()):
    text = str(text)
    for secret in secrets:
        if secret:
            for value in {secret, quote(secret, safe=""), quote_plus(secret)}:
                text = text.replace(value, "REDACTED")
    text = re.sub(r"(?i)(OC(?:=|%3D))[^&\s<>\"']+", r"\1REDACTED", text)
    return text


def clean_xml(raw):
    if b"<!DOCTYPE" in raw.upper() or b"<!ENTITY" in raw.upper():
        raise ValueError("XML_DTD_OR_ENTITY")
    root = ET.fromstring(raw)
    # Namespace is harmless formatting; original bytes remain separately hashed.
    for elem in root.iter():
        elem.tag = elem.tag.rsplit("}", 1)[-1]
    return root


def first(root, name):
    return (root.findtext(".//" + name) or "").strip()


def normalized(text):
    return re.sub(r"\s+", "", text).replace("·", "ㆍ")


def inspect_xml(raw, kind, expected_name="", expected_id=""):
    root = clean_xml(raw)
    if root.tag.lower() in {"html", "error", "errors"}:
        raise ValueError("HTML_OR_API_ERROR")
    for tag in ("error", "errorCode", "errCode", "오류메시지"):
        if first(root, tag):
            raise ValueError("API_ERROR_XML")
    if kind in ("law_list", "admrul_list"):
        law = kind == "law_list"
        required_root = "LawSearch" if law else "AdmRulSearch"
        if root.tag != required_root:
            raise ValueError("UNEXPECTED_XML_ROOT")
        rows = []
        for el in root.iter("law" if law else "admrul"):
            rows.append({
                "name": first(el, "법령명한글" if law else "행정규칙명"),
                "id": first(el, "법령ID" if law else "행정규칙ID"),
                "serial": first(el, "법령일련번호" if law else "행정규칙일련번호"),
                "current": first(el, "현행연혁코드" if law else "현행연혁구분"),
                "effective_date": first(el, "시행일자"),
            })
        total = first(root, "totalCnt")
        if not total.isdigit() or (int(total) > 0 and not rows):
            raise ValueError("LIST_SCHEMA_MISMATCH")
        return {"status": "LIST_OK" if rows else "EMPTY_LIST", "root": root.tag,
                "total_count": int(total), "rows": rows,
                "note": "목록 수신은 본문 수신/전체 수집 성공과 별개"}
    if kind in ("law_body", "admrul_body"):
        law = kind == "law_body"
        if root.tag != ("법령" if law else "AdmRulService"):
            raise ValueError("UNEXPECTED_XML_ROOT")
        name = first(root, "법령명_한글" if law else "행정규칙명")
        doc_id = first(root, "법령ID" if law else "행정규칙일련번호")
        contents = ["".join(x.itertext()).strip() for x in root.iter("조문내용")]
        if not name or not any(contents) or not doc_id:
            raise ValueError("BODY_MISSING_NAME_ID_OR_ARTICLE")
        if expected_name and normalized(name) != normalized(expected_name):
            raise ValueError("BODY_NAME_MISMATCH")
        if expected_id and doc_id.lstrip("0") != expected_id.lstrip("0"):
            raise ValueError("BODY_ID_MISMATCH")
        return {"status": "BODY_OK", "root": root.tag, "document_name": name, "id": doc_id,
                "effective_date": first(root, "시행일자"),
                "promulgated_date": first(root, "공포일자" if law else "발령일자"),
                "article_blocks": len(contents), "article_characters": sum(map(len, contents)),
                "annex_links": sum(1 for x in root.iter() if x.tag in
                                   {"별표서식파일링크", "첨부파일링크"} and x.text),
                "sample": next(x for x in contents if x)[:240]}
    if kind == "notice":
        # The relay response contract is not supplied. XML alone cannot certify data.
        return {"status": "NOTICE_XML_REVIEW_REQUIRED", "root": root.tag,
                "child_tags": [e.tag for e in list(root)[:15]],
                "note": "예고 API XML 수신만 확인. 목록/0건/오류 판정은 실제 명세와 대조 필요"}
    if kind == "ip":
        raise ValueError("IP_RESPONSE_MUST_BE_JSON")
    raise ValueError("UNKNOWN_RESPONSE_KIND")


class RelayCheck:
    def __init__(self, config, law_oc, making_oc="", cookie="", session=None):
        self.config = dict(config)
        self.oc = law_oc.strip()
        self.making_oc = making_oc.strip()
        self.cookie = cookie.strip()
        self.secrets = (self.oc, self.making_oc, self.cookie)
        if not self.oc or self.oc.lower() == "test":
            raise ValueError("등록된 LAW_OC를 입력하세요. test 예제값은 사용하지 않습니다.")
        if any("\n" in x or "\r" in x for x in self.secrets):
            raise ValueError("인증 입력에 줄바꿈이 있습니다.")
        self.mode = config.get("mode", "relay")
        if self.mode not in ("relay", "direct_public"):
            raise ValueError("mode는 relay 또는 외부 검증용 direct_public만 가능합니다.")
        self.verify = config.get("ca_bundle") or True
        if config.get("ca_bundle") is False or (self.verify is not True and not Path(self.verify).is_file()):
            raise ValueError("CA 파일 경로를 확인하세요. 인증서 검증 해제는 지원하지 않습니다.")
        self.origin = config["relay_origin"].rstrip("/")
        p = urlsplit(self.origin)
        if p.scheme != "https" or not p.hostname or p.username or p.password or p.path or p.query or p.fragment:
            raise ValueError("relay_origin은 https://승인호스트:포트 형식입니다.")
        self.law_origin = config.get("law_origin", "http://www.law.go.kr").rstrip("/")
        if self.law_origin not in ("http://www.law.go.kr", "https://www.law.go.kr"):
            raise ValueError("법제처 원본 호스트 불일치")
        self.making_origin = config.get("making_origin", "https://opinion.lawmaking.go.kr").rstrip("/")
        if self.making_origin not in ("https://opinion.lawmaking.go.kr", "https://www.lawmaking.go.kr"):
            raise ValueError("입법센터 원본 호스트 불일치")
        self.session = session or requests.Session()
        if session is None:
            self.session.mount("https://", requests.adapters.HTTPAdapter(max_retries=0))
            self.session.mount("http://", requests.adapters.HTTPAdapter(max_retries=0))
        self.run_id = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime()) + "_" + uuid.uuid4().hex[:8]
        self.out = Path(config.get("output_root", "law_api_check_results")) / self.run_id
        self.out.mkdir(parents=True, exist_ok=False, mode=0o700)
        self.records = {}
        self.started = time.monotonic()
        self.last_call = 0.0
        self.calls = 0
        self.blocked = False
        self._save_summary()

    def _write(self, name, raw):
        path = self.out / name
        with path.open("xb") as handle:
            handle.write(raw)
        path.chmod(0o600)

    def _save_summary(self):
        summary = {"version": VERSION, "run_id": self.run_id, "transport": self.mode,
                   "python": platform.python_version(), "executable": sys.executable,
                   "requests_version": requests.__version__, "physical_calls": self.calls,
                   "max_calls": MAX_CALLS, "automatic_retries": 0,
                   "resume_available": False, "records": list(self.records.values()),
                   "scope": "소량 API 연결 진단. RAW 반입/MAIN/GENOS 배포는 수행하지 않음"}
        tmp = self.out / "summary.tmp"
        tmp.write_bytes(json_bytes(summary))
        tmp.chmod(0o600)
        tmp.replace(self.out / "summary.json")

    def request(self, name, path, target, params, kind, expected_name="", expected_id=""):
        if not re.fullmatch(r"[a-z0-9_]+", name):
            raise ValueError("Invalid diagnostic name")
        if path not in {"lawSearch.do", "lawService.do", "ogLmPp.xml", "ogLmPpMod.xml", "ptcpAdmPp.xml", "ip"}:
            raise ValueError("허용되지 않은 API 경로")
        if path.startswith("law") and target not in {"law", "eflaw", "admrul"}:
            raise ValueError("허용되지 않은 target")
        if any(k.lower() in {"oc", "target", "target_url", "cookie", "authorization"} for k in params):
            raise ValueError("인증과 target은 params에 직접 넣지 않습니다.")
        sig = digest(json_bytes([path, target, params, kind, expected_name, expected_id]))
        if name in self.records:
            if self.records[name]["request_signature"] != sig:
                raise ValueError("같은 이름의 요청 조건이 변경되었습니다. 새 진단을 시작하세요.")
            return self.records[name]  # success AND failure: do not blindly retry a cell
        row = {"name": name, "request_signature": sig, "target": target,
               "params": params, "transport": self.mode, "status": "PENDING",
               "started_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}
        self.records[name] = row
        if self.blocked or self.calls >= MAX_CALLS or time.monotonic() - self.started > 900:
            row["status"] = "BLOCKED_OR_BUDGET_EXHAUSTED"
            self._save_summary()
            return row
        is_making = path.endswith(".xml")
        if is_making and not self.making_oc:
            row["status"] = "MISSING_MAKING_OC"
            self._save_summary()
            return row
        arguments = dict(params)
        method, body = "GET", None
        headers = {"Accept": "application/xml, text/xml, application/json"}
        if path == "ip":
            if self.mode != "relay":
                raise ValueError("IP 조회는 사내 relay 모드에서만 실행")
            url = self.config.get("ip_lookup_url", "https://45.1.81.80:8443/oap/common/query/ip")
            p = urlsplit(url)
            if p.scheme != "https" or not p.hostname or p.username or p.password or p.query or p.fragment or p.path != "/oap/common/query/ip":
                raise ValueError("운영계 IP 조회 API 주소를 확인하세요.")
            method, body = "POST", {"EXTR_INSN_SECT_CODE": str(params.get("EXTR_INSN_SECT_CODE", "0"))}
            arguments = None
            row["original_url"] = url
        else:
            upstream = self.making_origin + "/rest/" + path if is_making else self.law_origin + "/DRF/" + path
            row["original_url"] = upstream + ("?target=" + target if target else "")
            arguments["OC"] = self.making_oc if is_making else self.oc
            if self.mode == "relay":
                url = self.origin + ("/law/making" if is_making else "/law/DRF")
                # target is placed ONCE: in header or query, never both.
                headers["target_url"] = upstream
                if target and self.config.get("target_in_header", True):
                    headers["target_url"] += "?target=" + target
                elif target:
                    arguments["target"] = target
            else:
                url = upstream.replace("http://", "https://", 1)
                if target:
                    arguments["target"] = target
        if self.cookie and self.mode == "relay":
            headers["Cookie"] = self.cookie
        time.sleep(max(0.0, 1.0 - (time.monotonic() - self.last_call)))
        self.last_call = time.monotonic()
        self.calls += 1
        row["physical_call"] = self.calls
        row["status"] = "IN_FLIGHT"
        self._save_summary()  # interrupt leaves an honest IN_FLIGHT record
        start = time.monotonic()
        raw = bytearray()
        try:
            with self.session.request(method, url, params=arguments, json=body, headers=headers,
                                      timeout=(8, 20), verify=self.verify,
                                      allow_redirects=False, stream=True) as response:
                row["http_status"] = response.status_code
                row["content_type"] = masked(response.headers.get("Content-Type", ""), self.secrets)
                for chunk in response.iter_content(65536):
                    if time.monotonic() - start > 45:
                        raise ValueError("RESPONSE_TIME_LIMIT")
                    if len(raw) + len(chunk) > MAX_BYTES:
                        raise ValueError("RESPONSE_SIZE_LIMIT")
                    raw.extend(chunk)
                if 300 <= response.status_code < 400:
                    raise ValueError("REDIRECT_BLOCKED")
                if response.status_code != 200:
                    if response.status_code in (401, 403, 429):
                        self.blocked = True
                    raise ValueError("HTTP_" + str(response.status_code))
                if kind == "ip":
                    parsed = json.loads(bytes(raw))
                    if not isinstance(parsed, (dict, list)):
                        raise ValueError("IP_JSON_SCHEMA_UNKNOWN")
                    row.update(status="IP_RESPONSE_REVIEW_REQUIRED",
                               note="실제 응답은 저장파일로 확인. 운영 주소 선택/성공코드/환경코드는 담당자 명세와 대조")
                else:
                    row.update(inspect_xml(bytes(raw), kind, expected_name, expected_id))
        except requests.exceptions.SSLError:
            self.blocked = True
            row.update(status="TLS_ERROR", reason="인증서/호스트명/CA 확인. 검증을 해제하지 마세요.")
        except requests.exceptions.Timeout:
            self.blocked = True
            row.update(status="TIMEOUT", reason="Dataiku 실행 서버→중계서버 네트워크/업스트림 시간 확인")
        except requests.exceptions.ConnectionError:
            self.blocked = True
            row.update(status="CONNECTION_ERROR", reason="DNS/방화벽/포트/프록시/실행 컨테이너 연결 확인")
        except requests.exceptions.RequestException as exc:
            row.update(status="TRANSPORT_ERROR", error_class=type(exc).__name__)
        except (ET.ParseError, UnicodeError, json.JSONDecodeError) as exc:
            row.update(status="PARSE_ERROR", error_class=type(exc).__name__, reason="XML 대신 HTML/로그인/JSON 포장/잘린 응답인지 확인")
        except ValueError as exc:
            row.update(status="REJECTED", reason=masked(str(exc), self.secrets))
        finally:
            row["elapsed_seconds"] = round(time.monotonic() - start, 3)
            row["received_bytes"] = len(raw)
            row["wire_sha256"] = digest(raw)
            if raw:
                # Search responses may echo OC inside detail URLs: never save that secret.
                clean = bytes(raw)
                for secret in self.secrets:
                    if secret:
                        for value in {secret, quote(secret, safe=""), quote_plus(secret)}:
                            clean = clean.replace(value.encode(), b"REDACTED")
                row["credential_redacted"] = clean != raw
                row["saved_sha256"] = digest(clean)
                row["response_file"] = name + ".response.gz"
                self._write(row["response_file"], gzip.compress(clean, mtime=0))
            safe_row = json.loads(masked(json.dumps(row, ensure_ascii=False), self.secrets))
            row.clear()
            row.update(safe_row)
            self._write(name + ".receipt.json", json_bytes(row))
            self._save_summary()
        return row

    def check_laws(self):
        law_name = self.config["law_name"]
        law_id = str(self.config["law_id"])
        if not law_id.isdigit():
            raise ValueError("law_id는 법령ID 숫자 문자열입니다.")
        # Can confirm the listed body API even if list forwarding is not yet registered.
        self.request("law_published_body", "lawService.do", "law", {"ID": law_id, "type": "XML"},
                     "law_body", law_name, law_id)
        self.request("law_search", "lawSearch.do", "law", {"query": law_name, "type": "XML", "display": "50", "page": "1"}, "law_list")
        adm = self.request("admrul_search", "lawSearch.do", "admrul",
                           {"query": self.config["admrul_name"], "type": "XML", "display": "50", "page": "1"}, "admrul_list")
        matches = [x for x in adm.get("rows", []) if x["current"] == "현행"
                   and normalized(x["name"]) == normalized(self.config["admrul_name"])]
        if len(matches) == 1 and matches[0]["serial"].isdigit():
            self.request("admrul_body", "lawService.do", "admrul", {"ID": matches[0]["serial"], "type": "XML"},
                         "admrul_body", self.config["admrul_name"], matches[0]["serial"])
        else:
            self.records["admrul_body"] = {"name": "admrul_body", "status": "SKIPPED_NO_UNIQUE_CURRENT_ID",
                                           "reason": "목록의 정확한 현행 항목 한 건이 필요. 임의 ID를 선택하지 않음"}
        # Current unpinned LawGoAdapter uses this endpoint; explicitly not a fallback.
        self.request("law_effective_body", "lawService.do", "eflaw", {"ID": law_id, "type": "XML"},
                     "law_body", law_name, law_id)
        self._save_summary()
        return list(self.records.values())

    def table(self):
        columns = ("name", "status", "http_status", "received_bytes", "effective_date", "reason")
        data = [{k: row.get(k, "") for k in columns} for row in self.records.values()]
        buf = io.StringIO()
        writer = csv.DictWriter(buf, fieldnames=columns)
        writer.writeheader()
        writer.writerows(data)
        path = self.out / "results.csv"
        path.write_text(buf.getvalue(), encoding="utf-8-sig")
        path.chmod(0o600)
        for row in data:
            print(" | ".join(str(row[k]) for k in columns))
        print("결과 위치:", self.out.resolve())
        return data

    def export_to_dataiku(self, folder_id):
        import dataiku
        if not folder_id:
            raise ValueError("Managed Folder ID를 입력하세요.")
        folder = dataiku.Folder(folder_id)
        uploaded = []
        # Use API streams for container/S3 compatibility; no get_path assumption.
        for path in sorted(self.out.iterdir()):
            if path.is_file() and path.suffix in {".json", ".csv", ".gz"}:
                remote = self.run_id + "/" + path.name
                folder.upload_data(remote, path.read_bytes())
                with folder.get_download_stream(remote) as stream:
                    if digest(stream.read()) != digest(path.read_bytes()):
                        raise RuntimeError("Managed Folder 업로드 해시 불일치: " + path.name)
                uploaded.append(remote)
        print("Managed Folder 저장 및 재읽기 검증:", len(uploaded), "파일 /", self.run_id)
        return uploaded

NOTEBOOK_CODE_SHA256 = '06842576d5a90324643faa5ac2cecb577bb300550015aef4321bc600beab1fbf'

"""Two-stage notebook coordinator around the unchanged, sealed RelayCheck."""
from contextlib import contextmanager
import fcntl


class NotebookRun:
    SCHEMA = "dataiku-relay-law-two-stage-v1"

    def __init__(self, config):
        self.config = dict(config)
        run_id = self.config["run_id"]
        if not re.fullmatch(r"[A-Za-z0-9_-]{1,64}", run_id):
            raise ValueError("run_id에는 영문·숫자·밑줄·하이픈만 사용하세요.")
        origin = self.config["relay_origin"].rstrip("/")
        parsed = urlsplit(origin)
        if (parsed.scheme != "https" or not parsed.hostname or parsed.username or parsed.password
                or parsed.path or parsed.query or parsed.fragment):
            raise ValueError("relay_origin은 https://호스트:포트 형식이어야 합니다.")
        if self.config["law_origin"] not in ("http://www.law.go.kr", "https://www.law.go.kr"):
            raise ValueError("법제처 원본 주소를 확인하세요.")
        if not str(self.config["law_id"]).isdigit() or not self.config["law_name"].strip():
            raise ValueError("법령 ID와 법령명을 확인하세요.")
        self.verify = self.config["ca_bundle"] or True
        if self.config["ca_bundle"] is False or (self.verify is not True and not Path(self.verify).is_file()):
            raise ValueError("ca_bundle은 실제 CA 파일 경로 또는 빈 문자열이어야 합니다.")
        self.url = origin + "/law/DRF"
        self.binding = digest(json_bytes({"config": self.config, "code": NOTEBOOK_CODE_SHA256}))
        self.out = Path(self.config["output_root"]) / run_id
        self.out.mkdir(parents=True, exist_ok=True, mode=0o700)
        self.path = self.out / "connection_result.json"
        with self.transaction():
            pass

    def ensure_config(self, config):
        if dict(config) != self.config:
            raise ValueError("설정이 바뀌었습니다. run_id도 바꾼 뒤 1단계부터 실행하세요.")

    def save(self):
        temporary = self.path.with_suffix(".tmp")
        temporary.write_bytes(json_bytes(self.state))
        temporary.chmod(0o600)
        temporary.replace(self.path)

    @contextmanager
    def transaction(self):
        with (self.out / "run.lock").open("a") as handle:
            try:
                fcntl.flock(handle.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
            except BlockingIOError:
                raise RuntimeError("같은 run_id가 실행 중입니다. 완료 후 결과를 확인하세요.") from None
            try:
                if self.path.exists():
                    self.state = json.loads(self.path.read_text(encoding="utf-8"))
                    if (self.state.get("schema") != self.SCHEMA or self.state.get("binding") != self.binding):
                        raise ValueError("저장된 실행과 코드·설정이 다릅니다. 기존 기록을 보존하고 run_id를 바꾸세요.")
                    if self.state.get("attempts") not in (0, 1, 2):
                        raise ValueError("저장된 요청 횟수가 올바르지 않습니다.")
                else:
                    self.state = {"schema": self.SCHEMA, "binding": self.binding,
                                  "created_at": time.time(), "deadline": time.time() + 900,
                                  "attempts": 0, "max_attempts": 2, "automatic_retries": 0,
                                  "relay": None, "law": None}
                    self.save()
                yield
            finally:
                fcntl.flock(handle.fileno(), fcntl.LOCK_UN)

    def begin(self, stage):
        existing = self.state[stage]
        if existing is not None:
            print("저장된 결과 사용 — 추가 요청 없음:", stage, existing["status"])
            if existing["status"] in ("IN_FLIGHT", "INTERRUPTED"):
                print("이전 요청의 완료 여부가 불명확합니다. 자동으로 다시 호출하지 않습니다.")
            return False
        if time.time() >= self.state["deadline"] or self.state["attempts"] >= 2:
            self.state[stage] = {"status": "BUDGET_EXHAUSTED", "reason": "15분 또는 총 2회 요청 한도"}
            self.save()
            return False
        self.state["attempts"] += 1
        self.state[stage] = {"status": "IN_FLIGHT", "started_at": time.time()}
        self.save()
        return True

    def preflight(self):
        with self.transaction():
            if not self.begin("relay"):
                return self.state["relay"]
            row = self.state["relay"]
            start = time.monotonic()
            session = requests.Session()
            session.mount("https://", requests.adapters.HTTPAdapter(max_retries=0))
            try:
                settings = session.merge_environment_settings(self.url, {}, True, self.verify, None)
                row["proxy_enabled"] = bool(requests.utils.select_proxy(self.url, settings["proxies"]))
                # HEAD has no OC, Cookie, target_url or upstream API parameters.
                with session.request("HEAD", self.url, timeout=(8, 10), verify=self.verify,
                                     allow_redirects=False, stream=True) as response:
                    row.update(status="HTTP_RESPONSE_RECEIVED", http_status=response.status_code,
                               reason="현재 중계 주소/프록시 경로에서 HTTP 응답 수신. 법제처 성공은 2단계에서 확인.")
            except requests.exceptions.SSLError:
                row.update(status="TLS_ERROR", reason="인증서·호스트명·CA를 확인하세요.")
            except requests.exceptions.ConnectTimeout:
                row.update(status="CONNECT_TIMEOUT", reason="중계 연결 단계 시간 초과")
            except requests.exceptions.ReadTimeout:
                row.update(status="READ_TIMEOUT", reason="HTTP 응답 대기 시간 초과")
            except requests.exceptions.Timeout:
                row.update(status="TIMEOUT", reason="연결 또는 응답 시간 초과")
            except requests.exceptions.ConnectionError as exc:
                row.update(status="CONNECTION_ERROR", error_class=type(exc).__name__,
                           reason="주소·포트·실행 노드 네트워크·프록시를 확인하세요.")
            except requests.exceptions.RequestException as exc:
                row.update(status="REQUEST_ERROR", error_class=type(exc).__name__)
            except KeyboardInterrupt:
                row.update(status="INTERRUPTED")
                raise
            except Exception as exc:
                row.update(status="LOCAL_ERROR", error_class=type(exc).__name__)
                raise RuntimeError("통신 확인 중 로컬 오류입니다. 저장된 오류 종류를 확인하세요.") from None
            finally:
                session.close()
                row["elapsed_seconds"] = round(time.monotonic() - start, 3)
                self.save()
            print("1단계:", row["status"], "HTTP", row.get("http_status", "응답 없음"))
            print(row.get("reason", ""))
            return row

    def law_ready(self):
        with self.transaction():
            if self.state["law"] is not None:
                print("2단계는 이미 시도했습니다. 결과 셀에서 저장된 결과를 확인하세요.")
                return False
            if not self.state["relay"] or self.state["relay"]["status"] != "HTTP_RESPONSE_RECEIVED":
                print("먼저 1단계에서 HTTP 응답을 확인하세요. 법제처 요청은 보내지 않았습니다.")
                return False
            if time.time() >= self.state["deadline"] or self.state["attempts"] >= 2:
                print("실행 한도가 끝났습니다. 새 확인이 필요하면 run_id를 바꿔 1단계부터 실행하세요.")
                return False
            return True

    def call_law(self, law_oc, cookie=""):
        # Credential validation occurs before reserving the only upstream attempt.
        if not law_oc.strip() or law_oc.strip().lower() == "test":
            raise ValueError("법제처에 등록된 OC를 입력하세요.")
        if any(c in law_oc + cookie for c in "\r\n"):
            raise ValueError("OC·Cookie에 줄바꿈이 있습니다.")
        with self.transaction():
            if self.state["law"] is not None:
                return self.state["law"]
            if not self.state["relay"] or self.state["relay"]["status"] != "HTTP_RESPONSE_RECEIVED":
                print("중계 HTTP 응답이 확인되지 않아 법제처 요청을 보내지 않았습니다.")
                return None
            if not self.begin("law"):
                return self.state["law"]
            client = None
            try:
                cfg = dict(self.config, mode="relay", output_root=str(self.out / "law_response"))
                client = RelayCheck(cfg, law_oc, cookie=cookie)
                self.state["law"]["receipt_folder"] = str(client.out)
                self.save()
                # Existing sealed transport and XML identity validation, one GET only.
                result = client.request("law_published_body", "lawService.do", "law",
                                        {"ID": str(cfg["law_id"]), "type": "XML"},
                                        "law_body", cfg["law_name"], str(cfg["law_id"]))
                self.state["law"] = dict(result, receipt_folder=str(client.out))
            except KeyboardInterrupt:
                self.state["law"]["status"] = "INTERRUPTED"
                raise
            except Exception as exc:
                self.state["law"].update(status="LOCAL_ERROR", error_class=type(exc).__name__)
                raise RuntimeError("법제처 확인 중 로컬 오류입니다. 기록된 오류 종류와 receipt_folder를 확인하세요.") from None
            finally:
                if client is not None:
                    client.session.close()
                self.save()
            row = self.state["law"]
            print("2단계:", row["status"], "HTTP", row.get("http_status", "응답 없음"))
            if row["status"] == "BODY_OK":
                print("법령명:", row["document_name"], "/ 법령 ID:", row["id"])
                print("조문 블록:", row["article_blocks"], "/ 수신 바이트:", row["received_bytes"])
            else:
                print("법령 본문 수신 성공으로 확인되지 않았습니다:", row.get("reason", row["status"]))
            return row

    def show(self):
        with self.transaction():
            print("단계 | 상태 | HTTP | 시간(초)")
            for name, title in (("relay", "1. 중계 통신"), ("law", "2. 법제처 본문")):
                row = self.state[name] or {"status": "NOT_RUN"}
                print(title, "|", row["status"], "|", row.get("http_status", "-"), "|", row.get("elapsed_seconds", "-"))
            law = self.state["law"] or {}
            if law.get("status") == "BODY_OK":
                print("확인 완료: 중계를 통해 요청한 법령의 XML 본문·ID·이름·조문을 확인했습니다.")
                print("본문 예시:", law.get("sample", ""))
            else:
                print("법제처 본문 확인은 아직 완료되지 않았습니다.")
            print("시도한 HTTP 요청:", self.state["attempts"], "/ 최대 2회")
            print("결과 JSON:", self.path.resolve())
            if law.get("receipt_folder"):
                print("본문/개별 기록:", law["receipt_folder"])
            return self.state


## 1. API 중계서버 통신 확인
`HEAD https://중계주소/law/DRF` 요청 1회. OC·Cookie·target_url은 보내지 않습니다.
HTTP 401·403·404·405도 **응답 수신**입니다. 법제처 호출 성공을 뜻하지 않습니다. 프록시가 있다면 프록시 경로 응답일 수 있으므로 실제 중계/법제처 처리 여부는 다음 단계에서 확인합니다.
시간 초과·TLS·연결 오류로 HTTP 응답이 없으면 2단계는 요청을 보내지 않습니다.


In [ ]:
# 법제처 인증 없이 중계 주소에 HEAD 요청을 1회 보냅니다.
run = NotebookRun(CONFIG)
print("요청: HEAD", run.url)
_relay_result = run.preflight()


## 2. 중계를 통해 법제처 법령 본문 호출
법제처에 등록된 **국가법령정보센터 OC**와 필요한 경우 현재 중계 Cookie를 입력합니다. 이전 입법센터 사진의 OC가 이 API에서도 유효하다고 가정하지 않습니다.
중계 `/law/DRF`에 GET을 보내고 `target_url` 헤더로 `lawService.do?target=law`를 전달합니다.


In [ ]:
# 1단계에서 HTTP 응답을 확인한 뒤, 같은 중계 주소로 법제처 GET 요청 1회.
from getpass import getpass
run.ensure_config(CONFIG)
if run.law_ready():
    print("요청: GET", run.url)
    print("target_url:", CONFIG["law_origin"] + "/DRF/lawService.do?target=law")
    _oc = getpass("법제처에 등록된 OC: ")
    _cookie = getpass("현재 중계 Cookie (필요 없으면 Enter): ")
    try:
        _law_result = run.call_law(_oc, _cookie)
    finally:
        del _oc, _cookie


## 3. 결과 확인
`BODY_OK`는 HTTP 200과 XML 법령명·ID·조문 확인을 모두 통과한 상태입니다. HTML 로그인 화면이나 오류 XML은 성공으로 처리하지 않습니다.


In [ ]:
run.ensure_config(CONFIG)
_summary = run.show()


### 결과 보존과 다시 확인
결과는 커널 실행 폴더의 `dataiku_relay_law_results/<run_id>/`에 저장됩니다. 본문은 인증값을 가린 압축 파일로 보존하며, Oracle/Managed Folder에 자동 적재하지 않습니다.

같은 run_id의 성공·실패·중단은 자동 재호출하지 않습니다. 커널 재시작 후에도 동일하게 처리합니다. 중단된 요청의 기록을 먼저 확인하고 새 시도가 필요하면 run_id를 변경하세요. 자동 재시도 0회, 한 실행에서 HEAD 1회 + GET 1회, 새 요청 시작 한도는 15분입니다.

사내 연결 성공은 이 노트북의 현장 실행 결과로 확인해야 합니다. 인증값을 입력해 실행한 뒤에는 공유 전에 셀 출력을 지우세요. 이 파일에는 인증값이 저장되어 있지 않습니다.
